In [3]:
cd /home/nampv1/projects/vnpost_asr/

/home/nampv1/projects/vnpost_asr


In [4]:
!pip install -r requirements.txt

  Cloning https://github.com/huggingface/transformers to /tmp/pip-req-build-wahr62xk
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers /tmp/pip-req-build-wahr62xk
  Resolved https://github.com/huggingface/transformers to commit 25b4a0d8aef515ed309c935607473da319d9291c
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 12.6 MB/s eta 0:00:00a 0:00:01
  Created wheel for transformers: filename=transformers-4.57.0.dev0-py3-none-any.whl size=12602925 sha256=ea2442a21ca81367b30045a00b7603e8afdd939bcd54ed03bc652597bcd8f7e1
  Stored in directory: /tmp/pip-ephem-wheel-cache-ok3oyasf/wheels/04/a3/f1/b88775f8e1665827525b19ac7590250f1038d947067beba9fb
Successfully built transformers
  Attempting uninstall: tokenizers
    Found existing i

In [19]:
print(dir(model))


['T_destination', '__annotations__', '__bool__', '__call__', '__class__', '__contains__', '__copy__', '__deepcopy__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattr__', '__getattribute__', '__getitem__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__iter__', '__jit_unused_properties__', '__le__', '__len__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__reduce_package__', '__repr__', '__setattr__', '__setstate__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_apply', '_backward_hooks', '_backward_pre_hooks', '_buffers', '_c', '_call_impl', '_compiled_call_impl', '_concrete_type', '_constants_set', '_construct', '_disable_script_meta', '_finalize_scriptmodule', '_forward_hooks', '_forward_hooks_always_called', '_forward_hooks_with_kwargs', '_forward_pre_hooks', '_forward_pre_hooks_with_kwargs', '_get_backward_hooks', '_get_backward_pre_hooks', '_get_name', '_initial

In [12]:
print(model.code)   # in ra toàn bộ TorchScript code để xem có những function nào
print(model.__dict__.keys())


AttributeError: 'RecursiveScriptModule' object has no attribute 'code'

In [28]:
import torch
from huggingface_hub import hf_hub_download
import torchaudio

# 1. Tải TorchScript model
jit_model_path = hf_hub_download(
    repo_id="zzasdf/viet_iter3_pseudo_label",
    filename="exp/jit_script.pt"
)
model = torch.jit.load(jit_model_path, map_location="cpu")
model.eval()

# 2. Đọc file audio
waveform, sr = torchaudio.load("/home/nampv1/projects/vnpost_asr/data/examples/example2.wav")
if sr != 16000:
    waveform = torchaudio.functional.resample(waveform, sr, 16000)

# 3. Tạo đặc trưng log-mel filterbank (80 chiều)
features = torchaudio.compliance.kaldi.fbank(
    waveform,
    num_mel_bins=80,
    frame_length=25,
    frame_shift=10,
    energy_floor=1.0,
    sample_frequency=16000
)

# [T, 80] → thêm batch dimension
features = features.unsqueeze(0)

# # 4. Chạy inference
# with torch.no_grad():
#     # tuỳ vào cách model export, có thể nó trả về logits hoặc trực tiếp sequence
#     output = model(features)

# print(type(output))
# print(output)


/home/nampv1/anaconda3/envs/asr/lib/python3.11/site-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(


In [29]:
features

tensor([[[-15.9424, -15.9424, -15.9424,  ..., -15.9424, -15.9424, -15.9424],
         [-15.9424, -15.9424, -15.9424,  ..., -15.9424, -15.9424, -15.9424],
         [-15.9424, -15.9424, -15.9424,  ..., -15.9424, -15.9424, -15.9424],
         ...,
         [-13.9441, -15.6648, -14.3491,  ...,  -9.1773,  -9.1688,  -8.9858],
         [-15.9424, -15.9424, -15.9424,  ..., -15.9424, -15.9424, -15.9424],
         [-15.9424, -15.9424, -15.9424,  ..., -15.9424, -15.9424, -15.9424]]])

In [30]:

# ===============================
# 3. Greedy decoding loop
# ===============================

# Lấy encoder output
with torch.no_grad():
    encoder_out = model.encoder(features, torch.tensor([features.size(1)]))
    encoder_out = encoder_out[0] if isinstance(encoder_out, (tuple, list)) else encoder_out

# Khởi tạo sequence với blank token (thường là 0)
sequence = [0]
blank_id = 0

# Dự đoán từng time step
decoded_tokens = []
for t in range(encoder_out.size(1)):
    enc_t = encoder_out[:, t:t+1, :]  # [1,1,enc_dim]
    
    # Decoder input: last token
    dec_input = torch.tensor([sequence[-1]]).unsqueeze(0)  # [1,1]
    dec_out = model.decoder(dec_input)  # [1,1,dec_dim]
    
    # Joiner
    logit = model.joiner(enc_t, dec_out)  # [1,1,vocab_size]
    pred = torch.argmax(logit, dim=-1).item()
    
    if pred != blank_id:
        decoded_tokens.append(pred)
        sequence.append(pred)
    else:
        sequence.append(blank_id)

# ===============================
# 4. Token → Text (symbol table)
# ===============================
# Bạn cần symbol table từ training, ví dụ:
# 0 = blank, 1='a', 2='b', ..., n=' '
# Dưới đây là ví dụ placeholder
# symbol_table = {1:'a', 2:'b', 3:'c', 4:' ', 0:''}
text = ''.join([symbol_table[t] for t in decoded_tokens if t != 0])

print("Predicted text:", text)

Predicted text: ▁NHỮNG 21▁SINH 131▁SINH 131▁VIÊN 152▁BỊ 59▁BỊ 59▁ĐIỂM 179▁ĐIỂM 179▁KÉM 1376▁KÉM 1376▁THƯỜNG 259▁THƯỜNG 259▁CHO 18▁CHO 18▁RẰNG 408▁RẰNG 408▁MÌNH 201▁MÌNH 201▁KHÔNG 29▁KHÔNG 29▁THÔNG 51▁THÔNG 51▁THÔNG 51▁MINH 242▁MINH 242▁NHƯNG 203▁NHƯNG 203▁NHƯNG 203▁RỒI 314▁RỒI 314▁RỒI 314▁HỌ 398▁HỌ 398▁SỚM 695▁SỚM 695▁NHẬN 137▁NHẬN 137▁RA 44▁RẰNG 408▁CÓ 10▁CÓ 10▁RẤT 120▁RẤT 120▁NHIỀU 53▁NHIỀU 53▁NGƯỜI 17▁NGƯỜI 17▁THÀNH 34▁THÀNH 34▁THÀNH 34▁CÔNG 11▁CÔNG 11▁TRONG 13▁TRONG 13▁CÁC 7▁CÁC 7▁NGÀNH 321▁NGÀNH 321▁CÔNG 11▁CÔNG 11▁KHỔ 1168▁KHỔ 1168▁KHỔ 1168▁KHỔ 1168NG 192▁L 594▁L 594Ồ 1015▁CŨNG 36▁CŨNG 36▁ĐÃ 14▁TỪNG 573▁TỪNG 573▁BỊ 59▁BỊ 59▁ĐIỂM 179▁ĐIỂM 179▁KÉM 1376▁KÉM 1376▁NHƯ 63▁NHƯ 63▁VẬY 362


In [24]:
# 1. Load tokens.txt
from huggingface_hub import hf_hub_download

token_file = hf_hub_download(
    repo_id="zzasdf/viet_iter3_pseudo_label",
    filename="data/Vietnam_bpe_2000_new/tokens.txt"
)

with open(token_file, "r", encoding="utf-8") as f:
    symbol_table = [line.strip() for line in f.readlines()]  # list: id → token

# Example: symbol_table[0] = blank, symbol_table[1] = first token, ...


In [25]:
symbol_table

['<blk> 0',
 '<sos/eos> 1',
 '<unk> 2',
 '▁MỘT 3',
 '▁VÀ 4',
 '▁MƯƠI 5',
 '▁LÀ 6',
 '▁CÁC 7',
 '▁HAI 8',
 '▁CỦA 9',
 '▁CÓ 10',
 '▁CÔNG 11',
 '▁NĂM 12',
 '▁TRONG 13',
 '▁ĐÃ 14',
 '▁VỚI 15',
 '▁ĐƯỢC 16',
 '▁NGƯỜI 17',
 '▁CHO 18',
 '▁ĐỂ 19',
 '▁AN 20',
 '▁NHỮNG 21',
 '▁CÁI 22',
 '▁TẠI 23',
 '▁DÂN 24',
 '▁THÌ 25',
 '▁ĐẾN 26',
 '▁VỀ 27',
 '▁BA 28',
 '▁KHÔNG 29',
 '▁HIỆN 30',
 '▁ĐỒNG 31',
 '▁BỘ 32',
 '▁NHÂN 33',
 '▁THÀNH 34',
 '▁NGÀY 35',
 '▁CŨNG 36',
 '▁TRÊN 37',
 '▁ĐỘNG 38',
 '▁QUAN 39',
 '▁ĐỐI 40',
 '▁TỈNH 41',
 '▁NÀY 42',
 '▁TỪ 43',
 '▁RA 44',
 '▁VIỆC 45',
 '▁ĐÓ 46',
 '▁TRĂM 47',
 '▁NƯỚC 48',
 '▁HỘI 49',
 '▁CHÍNH 50',
 '▁THÔNG 51',
 '▁CƠ 52',
 '▁NHIỀU 53',
 '▁NAM 54',
 '▁NGHÌN 55',
 '▁Ở 56',
 '▁ĐỊNH 57',
 '▁CHÍN 58',
 '▁BỊ 59',
 '▁SỐ 60',
 '▁VỤ 61',
 '▁QUỐC 62',
 '▁NHƯ 63',
 '▁BỐN 64',
 '▁VÀO 65',
 '▁MÀ 66',
 '▁ĐIỀU 67',
 '▁PHÁT 68',
 '▁THÁNG 69',
 '▁LÀM 70',
 '▁THEO 71',
 '▁NHÀ 72',
 '▁KHI 73',
 '▁HÀNH 74',
 '▁SỰ 75',
 '▁THỰC 76',
 '▁THỜI 77',
 '▁TRUNG 78',
 '▁SẼ 79',
 '▁TRƯỜNG 80',
 '▁

In [ ]:
# !pip install torchaudio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 8.4 MB/s eta 0:00:00a 0:00:01
